In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import zipfile
import shutil

# Klasörleri oluştur
os.makedirs('/content/dataset/images', exist_ok=True)
os.makedirs('/content/dataset/labels', exist_ok=True)

# Zipleri çıkar
with zipfile.ZipFile('/content/drive/MyDrive/sigara_verisi.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/temp_sigara')
with zipfile.ZipFile('/content/drive/MyDrive/cam_verisi.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/temp_cam')
with zipfile.ZipFile('/content/drive/MyDrive/duman_verisi.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/temp_duman')

def dataset_entegre_et(temp_yol, önek, yeni_id):
    img_dir, lbl_dir = None, None
    for root, dirs, files in os.walk(temp_yol):
        if 'images' in root: img_dir = root
        if 'labels' in root: lbl_dir = root

    if not img_dir or not lbl_dir: return

    for dosya_adi in os.listdir(lbl_dir):
        if dosya_adi.endswith('.txt'):
            base_name = os.path.splitext(dosya_adi)[0]
            img_ext = next((ext for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.PNG']
                            if os.path.exists(os.path.join(img_dir, base_name + ext))), None)

            if img_ext:
                eski_txt = os.path.join(lbl_dir, dosya_adi)
                yeni_txt = os.path.join('/content/dataset/labels', f"{önek}_{dosya_adi}")

                with open(eski_txt, 'r') as f: satirlar = f.readlines()
                yeni_satirlar = [f"{yeni_id} " + " ".join(satir.split()[1:]) + "\n" for satir in satirlar if satir.split()]
                with open(yeni_txt, 'w') as f: f.writelines(yeni_satirlar)

                shutil.copy(os.path.join(img_dir, base_name + img_ext),
                            os.path.join('/content/dataset/images', f"{önek}_{base_name}{img_ext}"))

print("Veriler birleştiriliyor...")
dataset_entegre_et('/content/temp_sigara', 'sigara', 0)
dataset_entegre_et('/content/temp_cam', 'cam', 1)
dataset_entegre_et('/content/temp_duman', 'duman', 2)

# Çöpleri temizle
shutil.rmtree('/content/temp_sigara')
shutil.rmtree('/content/temp_cam')
shutil.rmtree('/content/temp_duman')
print(f"Birleştirme Tamam! Toplam Resim: {len(os.listdir('/content/dataset/images'))}")

Veriler birleştiriliyor...
Birleştirme Tamam! Toplam Resim: 6318


In [ ]:
import os
import random
import shutil

base_dir = '/content/dataset'
img_dir = os.path.join(base_dir, 'images')
lbl_dir = os.path.join(base_dir, 'labels')

# Yeni 3'lü YOLO klasör yapılarını oluştur
os.makedirs(os.path.join(base_dir, 'train/images'), exist_ok=True)
os.makedirs(os.path.join(base_dir, 'train/labels'), exist_ok=True)
os.makedirs(os.path.join(base_dir, 'val/images'), exist_ok=True)
os.makedirs(os.path.join(base_dir, 'val/labels'), exist_ok=True)
os.makedirs(os.path.join(base_dir, 'test/images'), exist_ok=True)
os.makedirs(os.path.join(base_dir, 'test/labels'), exist_ok=True)

# Karıştır
images = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
random.shuffle(images)

# %70 - %20 - %10 indekslerini hesapla
train_split = int(len(images) * 0.70)
val_split = int(len(images) * 0.90) # 70 + 20 = 90

def move_files(file_list, split_type):
    for img_name in file_list:
        shutil.move(os.path.join(img_dir, img_name), os.path.join(base_dir, f'{split_type}/images', img_name))
        txt_name = os.path.splitext(img_name)[0] + '.txt'
        txt_path = os.path.join(lbl_dir, txt_name)
        if os.path.exists(txt_path):
            shutil.move(txt_path, os.path.join(base_dir, f'{split_type}/labels', txt_name))

# Dosyaları taşı
move_files(images[:train_split], 'train')
move_files(images[train_split:val_split], 'val')
move_files(images[val_split:], 'test')

# Çöpleri temizle
shutil.rmtree(img_dir)
shutil.rmtree(lbl_dir)
print(f"70-20-10 Bölme Başarılı!")
print(f"Train: {train_split} | Valid: {val_split - train_split} | Test: {len(images) - val_split}")

70-20-10 Bölme Başarılı!
Train: 4422 | Valid: 1264 | Test: 632


In [ ]:
yaml_icerik = """
path: /content/dataset
train: train/images
val: val/images
test: test/images

names:
  0: izmarit
  1: cam_kirigi
  2: duman
"""

with open('/content/dataset/data.yaml', 'w') as f:
    f.write(yaml_icerik.strip())
print("Test klasörünü de içeren data.yaml dosyası başarıyla oluşturuldu!")

Test klasörünü de içeren data.yaml dosyası başarıyla oluşturuldu!


In [ ]:
# Kütüphaneyi kur
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.7 MB/s eta 0:00:00


In [ ]:
# Eğitimi ateşle
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

model.train(
    data='/content/dataset/data.yaml',
    epochs=50,
    imgsz=640,
    project='/content/drive/MyDrive/Yangin_Model_Projesi',
    name='drone_erken_uyari'
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgs

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f26c4178b00>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04